# 19 · `gl_engine/cli.py`

## What this file is for

Driving all of it from a terminal — five verbs, and none of them rate.

That is the interesting constraint. The CLI is for **looking at what resolves and proving it**: which packages govern, which countrywide editions are live, what one table holds, whether the content is consistent, and what the corpus contains.

**Depends on:** everything. It is a thin shell over the modules above.

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine import cli

for name, obj in vars(cli).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != cli.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Call `main()` directly with the arguments you'd type.

In [ ]:
from gl_engine import cli

code = cli.main(["resolve", "GA", "20260811"])
print("\nexit code:", code)

## The interesting case

### Which countrywide editions are live at once

In [ ]:
cli.main(["parents", "20260811"])

The same fact notebook 05 measured, from a shell. More than one national edition being live is normal, and it is why the declared-parent rule exists.

### Looking inside one table

In [ ]:
cli.main(["table", "GA", "20260811", "DedFactorProdsCSL", "--rows", "8"])

### The five verbs

In [ ]:
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    try:
        cli.main(["--help"])
    except SystemExit:
        pass
print(buf.getvalue()[:900])

## What it refuses

A bad argument exits non-zero rather than guessing what you meant.

In [ ]:
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    try:
        code = cli.main(["resolve", "ZZ", "20260811"])
    except SystemExit as e:
        code = e.code
    except Exception as e:
        code = type(e).__name__
print("result:", code)
print(buf.getvalue()[:300])

## Try it yourself

1. Run `census` and compare its numbers with what notebook 04 measured.
2. `check --deep` takes about 95 seconds. Run it once and read every assertion it prints.
3. Add a sixth verb that prints the schema for a jurisdiction. Which module does it need? (Notebook 13.)

In [ ]:
# your turn